In [1]:

## imports

import os
import sys
from pathlib import Path

import yaml
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Make the project root importable in a notebook context
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag.embedding_models.instance import embedding_manager
from src.rag.vector_store.instance import vector_store
from src.rag.embedding_models.embedding_factory import EmbeddingFactory
from src.rag.chunking_strategy.contextual_chunking.contextual_chunking import Contextualizer
from src.rag.llms.llama import LlamaManager


C:\Users\gabri\AppData\Local\Temp\ipykernel_14116\1436314504.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
currentDirectory = os.getcwd()
currentDirectory

'c:\\Users\\gabri\\Documents\\capstone-projects\\airline-support-bot\\backend\\rag-service\\notebooks'

In [3]:
## load data from the knowledge base 
loader = DirectoryLoader("../data/raw/",glob="**/*.md",loader_cls=TextLoader)
documents = loader.load()

print(f"number of documents :{len(documents)}")


number of documents :30


In [4]:
print(f"document one:{documents[0]}")
print("Content:\n", documents[0].page_content)
print("Metadata:\n", documents[0].metadata)


document one:page_content='---
document_id: KQ-AIRPORT-001
title: Airport Lounge Services
origin: KENYA AIRWAYS
domain: AIRPORT_SERVICES
category: LOUNGE_SERVICES
document_type: POLICY

applicable_to:
  - CUSTOMER
  - CUSTOMER_SERVICE_AGENT

access: PUBLIC
status: ACTIVE

language: EN
---

# Airport Lounge Services

Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.

## Kenya Airways Lounges

Kenya Airways operates the following lounges:

- Pride Lounge
- Simba Lounge
- Asante Lounge
- Msafiri Lounge

## Lounge Access Eligibility

Passengers eligible for Kenya Airways lounge access include:

- Kenya Airways Business Class passengers.
- SkyTeam Platinum and Gold cardholders.
- Eligible passengers travelling on Kenya Airw

In [5]:
## adding extra metadata to the documents
for document in documents:
    content = document.page_content

    # Split the front matter from the Markdown content
    if content.startswith("---"):
        _, front_matter, markdown = content.split("---", 2)

        # Convert YAML front matter into a Python dictionary
        metadata = yaml.safe_load(front_matter)

        # Add your metadata to LangChain's existing metadata
        document.metadata.update(metadata)

        # Remove the metadata from the actual page content
        document.page_content = markdown

In [6]:
## creating chunks from the documents
## 1. Creating the text splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "section"),
        ("##", "subsection"),
        ("###", "subsubsection")
    ]
)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [7]:

chunks = []

for document in documents:

    # First split according to Markdown headings
    sections = markdown_splitter.split_text(
        document.page_content
    )

    for section in sections:

        # Carry the original document metadata into the section
        section.metadata.update(document.metadata)

        # Split the section further if it is too large
        smaller_chunks = recursive_splitter.split_documents(
            [section]
        )

        for chunk in smaller_chunks:

            # Give every final chunk a unique ID
            chunk.metadata["chunk_id"] = (
                f"{document.metadata.get('document_id', 'UNKNOWN')}"
                f"-chunk-{len(chunks) + 1:03d}"
            )

            chunks.append(chunk)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 279


In [8]:
for chunk in chunks:
    print(f"Chunk ID: {chunk.metadata['chunk_id']}")
    print(f"Chunk Content: {chunk.page_content}")  # Print first 100 characters
    print(f"Chunk Metadata: {chunk.metadata}")
    


Chunk ID: KQ-AIRPORT-001-chunk-001
Chunk Content: Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.
Chunk Metadata: {'section': 'Airport Lounge Services', 'source': '..\\data\\raw\\airport_services\\lounge_services.md', 'document_id': 'KQ-AIRPORT-001', 'title': 'Airport Lounge Services', 'origin': 'KENYA AIRWAYS', 'domain': 'AIRPORT_SERVICES', 'category': 'LOUNGE_SERVICES', 'document_type': 'POLICY', 'applicable_to': ['CUSTOMER', 'CUSTOMER_SERVICE_AGENT'], 'access': 'PUBLIC', 'status': 'ACTIVE', 'language': 'EN', 'chunk_id': 'KQ-AIRPORT-001-chunk-001'}
Chunk ID: KQ-AIRPORT-001-chunk-002
Chunk Content: Kenya Airways operates the following lounges:  
- Pride Lounge
- Simba Lounge
- Asante Lounge
- Msafiri Lounge
Chunk Me

In [9]:
llm = LlamaManager()
contextualizer = Contextualizer(llm)

In [10]:
for i, chunk in enumerate(chunks):

    current_document_id = chunk.metadata.get("document_id")

    # Previous chunk
    previous_chunk = None

    if i > 0:
        candidate = chunks[i - 1]

        if candidate.metadata.get("document_id") == current_document_id:
            previous_chunk = candidate

    # Next chunk
    next_chunk = None

    if i < len(chunks) - 1:
        candidate = chunks[i + 1]

        if candidate.metadata.get("document_id") == current_document_id:
            next_chunk = candidate

    # Generate context and enrich the chunk
    chunk = contextualizer.enrich_chunk(
        chunk,
        previous_chunk,
        next_chunk
    )

In [11]:
for chunk in chunks[:5]:
    print("=" * 80)
    print(chunk.page_content)

Context: The TARGET CHUNK provides an overview of Kenya Airways' lounge services at Jomo Kenyatta International Airport, building on the previous chunk's introduction to airport lounge services. It then lists the specific lounges operated by Kenya Airways, further elaborating on the airline's lounge offerings.

Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.
Context: The TARGET CHUNK provides a list of lounges operated by Kenya Airways, building on the previous section's introduction to airport lounge services and further elaborating on the airline's lounge offerings. It serves as a concrete example of the lounges mentioned in the previous section, allowing readers to visualize the specific lounges available.

Kenya 

In [12]:
embedding_manager =EmbeddingFactory.create_embedding_model("gemini")  # or "gemin" for the other model
embeddings =  embedding_manager.create_embeddings(chunks)


Processing chunks 1 to 100
Embeddings returned: 100
Batch completed. Waiting 1 second...
Processing chunks 101 to 200
Embeddings returned: 100
Batch completed. Waiting 1 second...
Processing chunks 201 to 279
Embeddings returned: 79


In [13]:
type(embeddings)
for emb in embeddings:
    print(f"Embedding: {emb[:10]}...") 

len(embeddings) # Print first 10 dimensions of the embedding

Embedding: [0.012806812, 0.021167418, 0.0019914983, -0.016305732, 0.012995039, -0.010798297, 0.01598594, -0.008646642, 0.0408059, -0.050820548]...
Embedding: [0.006418746, 0.013531205, -0.0053013456, -0.025927205, 0.01573943, -0.014770874, 0.016556555, -0.015358219, 0.034298208, -0.051924158]...
Embedding: [0.0072526266, 0.0039663403, -0.0029956903, -0.039167818, 0.002308131, -0.0063488376, 0.009826989, -0.020687284, 0.0147251785, -0.0571883]...
Embedding: [0.0053252066, 0.008660231, -0.0029787086, -0.030821152, 0.016371852, -0.011092805, 0.008617818, -0.0044599636, 0.01988785, -0.048763506]...
Embedding: [0.01337369, 0.019585792, 0.00744594, -0.027963566, 0.0049719764, -0.021122888, -0.006391791, 0.0011028384, 0.025626348, -0.051342532]...
Embedding: [0.0058691516, 0.021061746, -0.008025974, -0.034059674, 0.009612891, -0.016969034, 0.008554369, 0.0023268817, 0.03149878, -0.050587412]...
Embedding: [0.016872147, 0.011016818, -0.007206255, -0.019216297, 0.02564529, 0.0027255658, 0.00730

279

In [14]:

vector_store.add_chunks(
   chunks,
   embeddings
)

In [ ]:
import google.genai

print(google.genai.__version__)